In [36]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from numpy import random
import cv2

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
print(os.listdir('/content/drive/MyDrive'))

['General presentation.gslides', 'SLIDE PRESENTATION:.gdoc', 'Photos', 'Em2.1bexbct.gdoc', 'amankhanal.mp4', 'filter3.1bei.gdoc', 'landslide4sense.zip', 'best_unet_model.pth']


In [39]:
!unzip -q "/content/drive/MyDrive/landslide4sense.zip" -d /content/

replace /content/landslide4sense/TestData/img/image_1.h5? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [40]:
# =============================================================================
# HELPER FUNCTION: Compute Topographic Features from DEM
# =============================================================================
# This is a plain Python function — no PyTorch needed here.
# It takes numpy arrays as input and returns numpy arrays as output.


def compute_topographic_features(
    dem, slope_deg, res=10.0, azimuth=315, angle_altitude=45
):

    # --- Step 1: Pad the DEM by 1 pixel on all sides ---
    # np.gradient() needs neighbors to compute derivatives at each pixel.
    # Without padding, edge pixels have missing neighbors and produce unstable results.
    # 'edge' mode repeats the border values outward.
    dem_padded = np.pad(dem, pad_width=1, mode="edge")

    # --- Step 2: Compute First-Order Gradients (Slope in X and Y) ---
    # np.gradient returns the rate of elevation change per meter.
    #   dx = slope going east  (columns)
    #   dy = slope going north (rows)
    # Note: np.gradient(array, spacing) returns (row_gradient, col_gradient)
    dy, dx = np.gradient(dem_padded, res)

    # --- Step 3: Compute Second-Order Gradients (Curvature) ---
    # Differentiate the first derivatives again to get curvature.
    #   d2x = rate of change of eastward slope  → east-west curvature
    #   d2y = rate of change of northward slope → north-south curvature
    d2y, _ = np.gradient(dy, res)  # We only need d2y from this call
    _, d2x = np.gradient(dx, res)  # We only need d2x from this call

    # --- Step 4: Crop back to original size ---
    # The padding added 1 row/column on each side, so we remove those borders
    # to get back to the original 128×128 shape.
    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    # --- Feature 1: Northness ---
    # aspect = compass direction the slope faces, in radians
    # We use arctan2(-dy, dx) because:
    #   - in image coordinates, the y-axis increases downward (south), not upward (north)
    #   - negating dy flips this to geographic north
    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)  # +1 = faces north, -1 = faces south

    # --- Feature 2: Curvature (Laplacian) ---
    # Positive curvature → convex terrain (ridges, hilltops)
    # Negative curvature → concave terrain (valleys, gullies)
    curvature = d2x + d2y

    # --- Feature 3: Hillshade ---
    # Simulates the brightness of terrain under a directional light source (the sun).
    # Uses the slope values from the raw ALOS band (slope_deg) instead of re-deriving slope.
    slope_rad = np.radians(slope_deg)
    azimuth_rad = np.radians(azimuth)
    altitude_rad = np.radians(angle_altitude)

    hillshade = np.sin(altitude_rad) * np.sin(slope_rad) + np.cos(
        altitude_rad
    ) * np.cos(slope_rad) * np.cos(azimuth_rad - aspect)
    hillshade = np.clip(hillshade, 0, 1)  # Clamp to [0, 1] — no negative light

    return northness, curvature, hillshade


def train_transform():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
            
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=(-0.1, 0.1),
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5
        ),
            
        ToTensorV2()
    ])
    
def val_transform():
    return A.Compose([
        ToTensorV2()
    ])


# =============================================================================
# PYTORCH DATASET CLASS: LandslideDataset
# =============================================================================
# In TensorFlow you often loaded data with tf.data.Dataset or keras utilities.
# In PyTorch, the standard way is to define a Dataset class with these 3 methods:
#
#   __init__  → runs once when you create the dataset object (setup/configuration)
#   __len__   → tells PyTorch how many samples exist (used by DataLoader)
#   __getitem__ → loads and returns ONE sample by index (called repeatedly during training)
#
# PyTorch's DataLoader will call __getitem__ automatically in batches.


class LandslideDataset(Dataset):

    def __init__(self, img_dir, mask_dir=None, transform=None, file_ids = None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        # Build a list of all .h5 filenames in the folder
        # Extract just the numbers from image filenames → [1, 2, 3, ...]
        if file_ids is not None:
            self.file_ids = file_ids
        else:
            self.file_ids = sorted(
                [
                    int(f.split("_")[1].split(".")[0])  # "image_1.h5" → 1
                    for f in os.listdir(img_dir)
                    if f.endswith(".h5")
                ]
            )

    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        PyTorch's DataLoader calls this internally to know when one epoch is complete.
        """
        return len(self.file_ids)

    def __getitem__(self, idx):
        """
        Loads and returns a single (image, mask) pair by index.
        """

        file_id = self.file_ids[idx]  # e.g. 1
        img_name = f"image_{file_id}.h5"  # → "image_1.h5"
        mask_name = f"mask_{file_id}.h5"  # → "mask_1.h5"

        # --- Load image ---
        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        # --- Load mask ---
        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)
        eps = 1e-6

        # --- Extract Required Raw Bands (0-indexed) ---
        # ALOS PALSAR band layout (0-indexed in the .h5 file):
        blue = raw_image[:, :, 1]  # Band 2  — Blue
        green = raw_image[:, :, 2]  # Band 3  — Green
        red = raw_image[:, :, 3]  # Band 4  — Red
        nir = raw_image[:, :, 7]  # Band 8  — Near Infrared
        swir1 = raw_image[:, :, 10]  # Band 11 — Shortwave Infrared 1
        slope = raw_image[:, :, 12]  # Band 13 — Slope (degrees)
        dem = raw_image[:, :, 13]  # Band 14 — Digital Elevation Model (meters)

        # --- Compute Topographic Features from DEM ---
        northness, curvature, hillshade = compute_topographic_features(dem, slope)
        # --- Compute Spectral Indices ---
        ndvi = (nir - red) / (nir + red + eps)

        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)

        ndwi = (green - nir) / (green + nir + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 8)
        image_8ch = np.stack(
            [dem, slope, northness, curvature, hillshade, ndvi, bsi, ndwi], axis=-1
        )  # Final shape: (128, 128, 8)

        if self.transform:
            
            augmented = self.transform(image=image_8ch, mask=mask)
            image = augmented['image'].float()
            mask = augmented['mask'].long()
        else:
            image_8ch = image_8ch.transpose((2, 0, 1))  # (C, H, W)

            image = torch.from_numpy(image_8ch).float()
            mask = torch.from_numpy(mask).long()

        return image, mask

In [41]:
# ----- Part-1: U-Net Architecture -----


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),  ## out_channels means filters in keras, and keras figures out the in_channel itself
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# Encoder Block: ConvBlock + MaxPool


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled
        # return both:
        # features -> will be passed accross via skip connection to decoder
        # pooled -> goes down to the next encoder block


# Decoder Block: Upsampe + Concatenate skip + convBlock
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        # convTransposed2d doubles the spatial size: (64, 64) → (128, 128)

        self.conv = ConvBlock(out_channels * 2, out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)  # concatenate along channel dimension
        x = self.conv(x)
        return x


# --- Full U-Net Model ---
class UNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=2):
        """
        in_chennels : number of input channels - 8 for our dataset
        num_classes : 2 for binary segmentation (landslide / no-landslide)
        """

        super().__init__()

        # Encoder ( Contracting Path)

        self.enc1 = EncoderBlock(in_channels, 64)  # 8 → 64
        self.enc2 = EncoderBlock(64, 128)  # 64 → 128
        self.enc3 = EncoderBlock(128, 256)  # 128 → 256
        self.enc4 = EncoderBlock(256, 512)  # 256 → 512

        # Bottleneck (deepest point - no pooling)

        self.bottleneck = ConvBlock(512, 1024)  # 512 → 1024

        # Decoder (Expanding Path)

        self.dec4 = DecoderBlock(1024, 512)  # 1024 → 512
        self.dec3 = DecoderBlock(512, 256)  # 512 → 256
        self.dec2 = DecoderBlock(256, 128)  # 256 → 128
        self.dec1 = DecoderBlock(128, 64)  # 128 → 64

        # --- Final Output Layer ---
        self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)
        # kernel size=1 -> 1x1 convolution, just maps 64 channels -> num_classes

    def forward(self, x):
        # ---Encoder ---
        skip1, x = self.enc1(x)  # skip for dec1, x: goes to enc2
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)  # skip for dec4, x: goes to bottleneck

        # --- Bottleneck ---
        x = self.bottleneck(x)

        # --- Decoder ---
        x = self.dec4(x, skip4)  # input from bottleneck, skip from enc4
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)  # input from dec2, skip from enc1

        # --- Final Output ---
        return self.output_conv(x)  # shape: (Batch, 2, 128, 128)

In [42]:
import torch
import torch.nn as nn

# =============================================================================
# PART 2: LOSS FUNCTION
# =============================================================================
# BACKGROUND CONTEXT:
# In landslide segmentation, we face extreme "Class Imbalance". A satellite patch 
# might have 16,000 background pixels and only 200 landslide pixels. 
# Standard CrossEntropy treats every pixel equally, which tempts the model to 
# lazily guess "No Landslide" everywhere to achieve 99% accuracy. 
# Dice Loss fixes this by ignoring background pixels and forcing the model to 
# focus entirely on maximizing the regional overlap (Intersection) with actual landslides.
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        # self.smooth prevents a catastrophic "Division by Zero" error.
        # If an image has 0 landslide pixels and the model correctly predicts 0,
        # the formula becomes 0/0 (which crashes). Adding 1.0 to top and bottom stabilizes it.
        self.smooth = smooth
 
    def forward(self, predictions, targets):
        # ---------------------------------------------------------------------
        # INPUT SHAPES:
        # predictions: (Batch, 2, H, W) -> Raw network outputs (Logits like [-2.4, 3.1])
        # targets:     (Batch, H, W)    -> Ground truth mask containing integers (0 or 1)
        # ---------------------------------------------------------------------
 
        # STEP 1: CONVERT LOGITS TO PROBABILITIES AND DROP THE BACKGROUND CHANNEL
        # 1. torch.softmax(..., dim=1) takes the raw channel values for every pixel 
        #    and crushes them into percentages (0.0 to 1.0) that sum to exactly 1.0 (100%).
        # 2. [:, 1, :, :] uses standard Python tuple indexing. It says:
        #    - First ':'  -> Keep all images in the batch.
        #    - '1'        -> Grab only Channel index 1 (the Landslide channel).
        #    - Last ':,:' -> Keep all Height rows and Width columns.
        # 3. DIMENSION DROPPING: Because we indexed with a single integer ('1') instead
        #    of a range ('1:2'), PyTorch drops the channel dimension entirely.
        #
        # FINAL SHAPE: (Batch, H, W) 
        # WHAT IS INSIDE: The decimal number stored inside each cell *is* the probability (0.0-1.0).
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :] 
        
        # Convert integer targets (0 and 1) to floats so we can do decimal multiplication with probs
        targets_f = targets.float()
 
        # STEP 2: CALCULATE THE INTERSECTION (OVERLAP)
        # Multiplying 'probs' (0.0 to 1.0) by 'targets_f' (0 or 1) acts as a mask.
        # If the target pixel is 0 (Background), the product becomes 0.
        # If the target pixel is 1 (Landslide), it preserves the model's probability score.
        # .sum(dim=(1, 2)) flattens and sums all pixel values across Height and Width.
        # OUTPUT SHAPE: (Batch,) -> One total intersection score per image in the batch.
        intersection = (probs * targets_f).sum(dim=(1, 2))
 
        # STEP 3: THE DICE COEFFICIENT FORMULA
        # Dice = (2 * |Intersection|) / (|Predictions| + |Targets|)
        # This measures the percentage of overlap between the predicted map and real map.
        # A perfect match yields a Dice Score of 1.0. Complete failure yields 0.0.
        # OUTPUT SHAPE: (Batch,)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        
        # STEP 4: RETURN THE LOSS
        # Deep learning optimizers are built to MINIMIZE a loss value toward 0.
        # Since we want to MAXIMIZE our Dice Score toward 1.0, we calculate Loss = 1 - Dice.
        # .mean() averages this loss score across all images in the batch.
        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    """Focal Loss for binary segmentation (landslide vs background).
       Down-weights easy background pixels and focuses on hard, rare landslide pixels."""
    def __init__(self, alpha=0.75, gamma=2.0, smooth=1e-6):
        # alpha = weight for positive class (landslide). Larger alpha = more focus on landslides.
        # gamma = focusing parameter; typical range [0.5, 5.0]. Higher gamma = more aggressive down-weighting of easy examples.
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth
        
    def forward(self, logits, targets):
        # logits: (B,2,H,W) raw outputs; targets: (B,H,W) with 0/1
        probs = torch.softmax(logits, dim=1)[:, 1]      # probability of landslide
        targets_f = targets.float()
        
        # p_t = probability of the true class (landslide if target==1, else 1-prob)
        pt = torch.where(targets_f == 1, probs, 1 - probs)
        
        # alpha_t = class‑dependent balancing factor: alpha for landslide, 1-alpha for background
        alpha_t = torch.where(targets_f == 1, self.alpha, 1 - self.alpha)
        
        # Focal loss per pixel: -α_t * (1-p_t)^γ * log(p_t)
        focal = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + self.smooth)
        
        return focal.mean()


class CombinedFocalDiceLoss(nn.Module):
    """Blend of Focal Loss (handles class imbalance) and Dice Loss (maximises region overlap).
       Often outperforms CE+Dice for extremely imbalanced tasks like landslide detection."""
    def __init__(self, focal_weight=0.5, dice_weight=0.5, alpha=0.75, gamma=2.0):
        super().__init__()
        self.focal = BinaryFocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        return (self.focal_weight * self.focal(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


class CombinedLoss(nn.Module):
    """Original combined loss: CrossEntropy + Dice.
       CE provides stable gradients early; Dice focuses on landslide overlap.
       This is a solid baseline before switching to Focal+Dice."""
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.dice = DiceLoss()
        
        # PyTorch's CrossEntropyLoss automatically accepts raw Logits (Batch, 2, H, W)
        # and integer Targets (Batch, H, W). It applies Softmax internally under the hood!
        self.ce   = nn.CrossEntropyLoss()
 
    def forward(self, predictions, targets):
        # WHY WE MIX THEM (50/50 Blend):
        # 1. CrossEntropy (CE) provides clean, highly stable mathematical gradients early 
        #    in training when the model is randomly guessing, but it suffers from class imbalance.
        # 2. Dice Loss ignores background pixels and focuses perfectly on landslide boundaries,
        #    but its gradients can be erratic and unstable when the model is completely untrained.
        # Combining them balances pixel-level confidence (CE) with global edge overlap (Dice).
        return (self.ce_weight   * self.ce(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))

In [43]:

# =============================================================================
# PART 3: METRICS
# =============================================================================
# What we track per epoch:
#   Loss    → how wrong the model is (lower is better)
#   IoU     → Intersection over Union — standard segmentation metric (higher is better)
#   F1      → same as Dice score — balances precision and recall (higher is better)
#   Precision → of all pixels predicted as landslide, how many actually are?
#   Recall    → of all actual landslide pixels, how many did we find?
 
def compute_metrics(predictions, targets, threshold=0.5):
    """
    predictions : (Batch, 2, H, W) raw logits from the model
    targets     : (Batch, H, W)    ground truth integer labels
    """
    # Get predicted class per pixel (0 or 1)
    pred_labels = torch.argmax(predictions, dim=1)   # shape: (Batch, H, W)
 
    # Flatten everything for easier computation
    pred = pred_labels.view(-1).cpu()
    true = targets.view(-1).cpu()
 
    # True Positives, False Positives, False Negatives
    tp = ((pred == 1) & (true == 1)).sum().float()
    fp = ((pred == 1) & (true == 0)).sum().float()
    fn = ((pred == 0) & (true == 1)).sum().float()
 
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    iou       = tp / (tp + fp + fn + 1e-6)
 
    return {
        "iou"      : iou.item(),
        "f1"       : f1.item(),
        "precision": precision.item(),
        "recall"   : recall.item()
    }
 
 

In [44]:

def train(img_dir, mask_dir, num_epochs, batch_size, lr, save_path):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # extracting and shuffling the file IDs
    all_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".h5")])
    all_ids = [int(f.split("_")[1].split(".")[0]) for f in all_files]
    
    # Shuffle with a fixed seed so train and val stays consistent
    random.seed(42)
    random.shuffle(all_ids)
    
    train_size = int(0.85 * len(all_ids))
    train_ids = all_ids[:train_size]
    val_ids = all_ids[train_size:]
    
    print(f"Total Samples: {len(all_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}")
    
    train_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=train_transform(), file_ids = train_ids)
    val_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=val_transform(), file_ids = val_ids)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size,  shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    
    # ----- Model, Loss and Optimizer ----- #
    model = UNet(in_channels=8, num_classes=2).to(device)
    criterion = CombinedFocalDiceLoss(focal_weight=0.5, dice_weight=0.5, alpha=0.75, gamma=2.0)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
    
    for epoch in range(1, num_epochs + 1):
        
        model.train()
        running_train_loss = 0.0
        
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()            # Clear old gradients
            predictions = model(images)      # Forward pass
            loss = criterion(predictions, targets) # Calculate loss
            loss.backward()                  # Backward pass (gradients)
            optimizer.step()                 # Update model weights
            
            running_train_loss += loss.item()
            
        train_loss = running_train_loss / len(train_loader)

        model.eval()
        running_val_loss = 0.0
        
        # Lists to temporarily hold metrics across validation batches
        val_iou, val_f1, val_prec, val_rec = [], [], [], []
        
        with torch.no_grad(): # Disable gradient engine to save memory
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                
                predictions = model(images)
                loss = criterion(predictions, targets)
                running_val_loss += loss.item()
                
                # Calculate metrics for this batch using your dictionary utility
                batch_metrics = compute_metrics(predictions, targets)
                val_iou.append(batch_metrics["iou"])
                val_f1.append(batch_metrics["f1"])
                val_prec.append(batch_metrics["precision"])
                val_rec.append(batch_metrics["recall"])

        val_loss = running_val_loss / len(val_loader)
        
        # Step the learning rate scheduler based on validation loss
        scheduler.step(val_loss)
        
        # Average the metrics over all validation batches
        val_metrics = {
            "iou": sum(val_iou) / len(val_iou),
            "f1": sum(val_f1) / len(val_f1),
            "precision": sum(val_prec) / len(val_prec),
            "recall": sum(val_rec) / len(val_rec)
        }

        # ---- Print Progress ----- #
        print(
            f"Epoch [{epoch:02d}/{num_epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f}  IoU: {val_metrics['iou']:.4f}  F1: {val_metrics['f1']:.4f}  "
            f"Precision: {val_metrics['precision']:.4f}  Recall: {val_metrics['recall']:.4f}"
        )
        
    # Save the final trained model weights to your E: drive or local path
    torch.save(model.state_dict(), save_path)
    print("Training Complete! Model Saved.")
    
    return model


if __name__ == "__main__":
    # 1. Corrected Cloud Paths (Updated with the landslide4sense parent folder)
    IMG_DIR  = "/content/landslide4sense/TrainData/img"
    MASK_DIR = "/content/landslide4sense/TrainData/mask"
    
    # 2. Permanent Cloud Storage (Saves the model weights directly to your Google Drive)
    SAVE_PATH = "/content/drive/MyDrive/best_unet_model.pth"

    # 3. Hyperparameters
    BATCH_SIZE = 16  
    EPOCHS     = 50
    LEARNING_RATE = 1e-4

    print("--- Starting Cloud Landslide Mapping Pipeline ---")
    
    # 4. Kick off training
    trained_model = train(
        img_dir    = IMG_DIR,
        mask_dir   = MASK_DIR,
        num_epochs = EPOCHS,
        batch_size = BATCH_SIZE,
        lr         = LEARNING_RATE,
        save_path  = SAVE_PATH
    )

--- Starting Cloud Landslide Mapping Pipeline ---
Using device: cuda
Total Samples: 3799 | Train: 3229 | Val: 570


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Epoch [01/50] | Train Loss: 0.4483 | Val Loss: 0.4210  IoU: 0.2760  F1: 0.4241  Precision: 0.2903  Recall: 0.8538
Epoch [02/50] | Train Loss: 0.4088 | Val Loss: 0.3922  IoU: 0.2919  F1: 0.4384  Precision: 0.3118  Recall: 0.8405
Epoch [03/50] | Train Loss: 0.3879 | Val Loss: 0.3671  IoU: 0.4568  F1: 0.6204  Precision: 0.5171  Recall: 0.7967
Epoch [04/50] | Train Loss: 0.3731 | Val Loss: 0.3477  IoU: 0.4991  F1: 0.6608  Precision: 0.5927  Recall: 0.7587
Epoch [05/50] | Train Loss: 0.3541 | Val Loss: 0.3197  IoU: 0.4485  F1: 0.6133  Precision: 0.5010  Recall: 0.8094
Epoch [06/50] | Train Loss: 0.3105 | Val Loss: 0.2471  IoU: 0.4372  F1: 0.6006  Precision: 0.5160  Recall: 0.7455
Epoch [07/50] | Train Loss: 0.2589 | Val Loss: 0.2120  IoU: 0.4590  F1: 0.6232  Precision: 0.6467  Recall: 0.6186
Epoch [08/50] | Train Loss: 0.2186 | Val Loss: 0.1930  IoU: 0.4712  F1: 0.6335  Precision: 0.5573  Recall: 0.7567
Epoch [09/50] | Train Loss: 0.2159 | Val Loss: 0.1953  IoU: 0.4696  F1: 0.6334  Precisio